# MP-20 PXRD Data Exploration
Visualize 5 random structures and their simulated Cu Kα PXRD patterns.
Sanity-check the data pipeline before any model code.

In [ ]:
import sys; sys.path.insert(0, '../src')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pymatgen.io.cif import CifParser
from pxrd_diff.simulator import PXRDSimulator

plt.rcParams.update({'figure.dpi': 120, 'font.size': 9})

In [ ]:
# Load cached patterns + metadata
two_theta = np.load('../data/cache/two_theta.npy')
test = np.load('../data/cache/test.npz', allow_pickle=True)
print(f"test set: {test['pattern'].shape[0]} structures, {test['pattern'].shape[1]} bins")
print(f"2theta range: [{two_theta[0]:.1f}, {two_theta[-1]:.1f}] deg")

# Space group distribution
sgs = test['spacegroup']
n_atoms = test['n_atoms']
print(f"Space groups: {len(np.unique(sgs))} unique, range [{sgs.min()}, {sgs.max()}]")
print(f"Atoms/cell: mean={n_atoms.mean():.1f}, median={np.median(n_atoms):.0f}, max={n_atoms.max()}")

In [ ]:
# Plot 5 random PXRD patterns
rng = np.random.default_rng(42)
idxs = rng.choice(len(test['pattern']), 5, replace=False)

fig, axes = plt.subplots(5, 1, figsize=(10, 12), sharex=True)
for ax, i in zip(axes, idxs):
    pat = test['pattern'][i]
    mid = str(test['material_id'][i])
    formula = str(test['formula'][i])
    sg = int(test['spacegroup'][i])
    na = int(test['n_atoms'][i])
    ax.plot(two_theta, pat, 'k-', lw=0.5)
    ax.fill_between(two_theta, pat, alpha=0.15, color='steelblue')
    ax.set_ylabel('I (norm)')
    ax.set_title(f'{mid}  {formula}  SG={sg}  N={na}', fontsize=9, loc='left')
    ax.set_ylim(0, 1.05)
axes[-1].set_xlabel(r'2$\theta$ (deg, Cu K$\alpha$)')
plt.tight_layout()
plt.savefig('../data/cache/sample_pxrd.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to data/cache/sample_pxrd.png')

In [ ]:
# Distribution plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.hist(sgs, bins=range(1, 232), color='steelblue', alpha=0.7)
ax1.set_xlabel('Space group number')
ax1.set_ylabel('Count')
ax1.set_title('Space group distribution (test set)')

ax2.hist(n_atoms, bins=range(1, 22), color='coral', alpha=0.7)
ax2.set_xlabel('Atoms per unit cell')
ax2.set_ylabel('Count')
ax2.set_title('Unit cell size distribution (test set)')

plt.tight_layout()
plt.savefig('../data/cache/distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Quick stats on pattern sparsity
pats = test['pattern']
nonzero_frac = (pats > 0.01).mean(axis=1)
print(f"Fraction of bins with I > 1%:")
print(f"  mean={nonzero_frac.mean():.3f}  min={nonzero_frac.min():.3f}  max={nonzero_frac.max():.3f}")
print(f"  -> patterns are ~{(1-nonzero_frac.mean())*100:.1f}% zero (sparse)")
print(f"Avg non-zero bins per pattern: {(pats > 0.01).sum(axis=1).mean():.0f} / {pats.shape[1]}")